# Table 3: NSEC/3 resource records per request
Reproduce Table 3 in *Dissecting NSEC and NSEC3 Deployment in the Wild* from the April 6, 2026 summary CSVs in `script/res/`. No measurement scripts are rerun.

For each population (Source or WWW), dataset, and mechanism (NSEC or NSEC3):

`percentage for k RRs = 100 × Count(k) / sum of Count over all RR-count categories`

`Count` is the number of queries with that many NSEC/3 records. Only NSEC/3 records are counted, excluding RRSIG and other auxiliary records. The denominator comes from the RR-count distribution itself. Requirements: Python 3, pandas, NumPy, and Jupyter. Run cells from top to bottom.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

DATE = "20260406"
REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "script/res" / f"{DATE}_rrcnt_stats.csv").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside dnsnsec26 or set REPO_ROOT explicitly.")
RES_DIR = REPO_ROOT / "script/res"
OUTPUT_DIR = REPO_ROOT / "notebook/results"
ZONES = ["com", "net", "org", "umbrella", "tranco", "radar", "majestic"]
LABELS = {"com": ".com", "net": ".net", "org": ".org",
          "umbrella": "Umbrella", "tranco": "Tranco",
          "radar": "Radar", "majestic": "Majestic"}

## Load and recompute the distributions
Inputs: `20260406_rrcnt_stats.csv` (Source) and `20260406_www_rrcnt_stats.csv` (WWW).

Recompute percentages from integer counts and check them against the saved `Perc` fractions. All categories remain in the denominator, even if the paper does not display them.

In [2]:
frames = []
for population, filename in [
    ("Source", f"{DATE}_rrcnt_stats.csv"),
    ("WWW", f"{DATE}_www_rrcnt_stats.csv"),
]:
    frame = pd.read_csv(RES_DIR / filename).rename(columns={
        "Zone": "zone", "Type": "mechanism", "cnt of RRs": "rr_count",
        "Count": "query_count", "Perc": "saved_fraction",
    })
    frame["population"] = population
    frames.append(frame)
stats = pd.concat(frames, ignore_index=True)
keys = ["population", "zone", "mechanism"]
if stats.duplicated(keys + ["rr_count"]).any():
    raise ValueError("Duplicate RR-count categories.")
expected_groups = {(population, zone, mechanism)
                   for population in ("Source", "WWW")
                   for zone in ZONES for mechanism in ("NSEC", "NSEC3")}
if set(map(tuple, stats[keys].to_numpy())) != expected_groups:
    raise ValueError("Expected both mechanisms for all seven datasets in both populations.")
for column in ["rr_count", "query_count"]:
    values = pd.to_numeric(stats[column], errors="raise")
    if values.isna().any() or (values < 0).any() or (values % 1 != 0).any():
        raise ValueError(f"{column} must contain nonnegative integers.")
    stats[column] = values.astype("int64")
if stats["rr_count"].lt(1).any():
    raise ValueError("RR counts must be positive.")
stats["total_queries"] = stats.groupby(keys)["query_count"].transform("sum")
if stats["total_queries"].le(0).any():
    raise ValueError("RR-distribution denominators must be positive.")
stats["fraction"] = stats["query_count"] / stats["total_queries"]
stats["percentage"] = 100 * stats["fraction"]
np.testing.assert_allclose(stats["fraction"], stats["saved_fraction"], rtol=1e-12, atol=1e-15)
np.testing.assert_allclose(stats.groupby(keys)["fraction"].sum(), 1.0, rtol=1e-12)
print("Recomputed fractions match all saved Perc values; each full distribution sums to 100%.")

Recomputed fractions match all saved Perc values; each full distribution sums to 100%.


## Display Table 3
The paper displays NSEC bins 1–3 and NSEC3 bins 1–4. Categories absent from a group's summary are shown as 0.00%. Percentages are rounded only for display; rounded rows may not sum to exactly 100%.

In [3]:
paper_columns = pd.MultiIndex.from_tuples([
    (population, mechanism, rr_count)
    for population in ("Source", "WWW")
    for mechanism, bins in (("NSEC", range(1, 4)), ("NSEC3", range(1, 5)))
    for rr_count in bins
], names=["Population", "Mechanism", "# RRs"])
table3 = stats.pivot(
    index="zone", columns=["population", "mechanism", "rr_count"], values="percentage"
).reindex(index=ZONES, columns=paper_columns).fillna(0.0)
table3.index = table3.index.map(LABELS)
table3.index.name = "Dataset"
formatted_table3 = table3.apply(lambda column: column.map(lambda value: f"{value:.2f}%"))
display(formatted_table3)

Population  Source                                                  WWW  \
Mechanism     NSEC                 NSEC3                           NSEC   
# RRs            1      2      3       1       2       3      4       1   
Dataset                                                                   
.com        97.85%  2.14%  0.01%  12.05%  44.61%  43.34%  0.00%  89.64%   
.net        96.66%  3.34%  0.01%  12.93%  46.01%  41.05%  0.00%  89.56%   
.org        97.06%  2.94%  0.00%  11.72%  44.72%  43.56%  0.00%  90.67%   
Umbrella    92.83%  7.17%  0.00%   4.91%  15.84%  79.25%  0.00%  94.33%   
Tranco      96.62%  3.38%  0.00%   4.06%  17.31%  78.63%  0.01%  95.90%   
Radar       97.52%  2.48%  0.00%   5.04%  18.63%  76.32%  0.01%  96.48%   
Majestic    96.74%  3.25%  0.01%   5.25%  19.77%  74.97%  0.01%  96.71%   

Population                                                
Mechanism                   NSEC3                         
# RRs            2      3       1       2       3      4  
Dataset                                                   
.com        10.35%  0.01%  11.94%  44.00%  44.06%  0.00%  
.net        10.44%  0.00%  12.78%  45.48%  41.74%  0.00%  
.org         9.33%  0.00%  11.57%  44.26%  44.16%  0.00%  
Umbrella     5.67%  0.00%   4.09%  14.86%  81.06%  0.00%  
Tranco       4.10%  0.00%   3.40%  16.99%  79.60%  0.00%  
Radar        3.52%  0.00%   4.21%  17.98%  77.81%  0.00%  
Majestic     3.29%  0.01%   4.57%  19.38%  76.05%  0.00%

## Export
`table3_stats.csv` preserves all bins, counts, denominators, and unrounded percentages. `table3_formatted.csv` reproduces the paper layout with three header rows. `table3_omitted_categories.csv` preserves categories not displayed in the paper. Rerunning this cell replaces these exports.

In [4]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
stats.to_csv(OUTPUT_DIR / "table3_stats.csv", index=False)
print(f"Saved Table 3 results to {OUTPUT_DIR}")

Saved Table 3 results to /u/yz6me/dnsnsec26/notebook/results
